In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
import random
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import albumentations as A
from torchvision.models import resnet50

In [2]:
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n Device : {device.upper()}')


 Device : CPU


In [3]:
TEST_PATH = '../input/aptos2019-blindness-detection/test.csv'
TEST_IMG = '../input/aptos2019-blindness-detection/test_images'
SAMPLE_SUB_PATH = '../input/aptos2019-blindness-detection/sample_submission.csv'

In [4]:
class AptosDataset(Dataset):
    def __init__(self, data_path, img_dir, name, transforms, resize = (512, 512)):
        self.data_path = data_path
        self.img_dir = img_dir
        self.resize = resize
        self.transforms = transforms
        self.df = pd.read_csv(self.data_path)
        self.test_size = int(np.round(0.3 * self.df.shape[0]))
        self.name = name
            
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):
        img_name = self.df['id_code'][idx] + '.png'
        img_path = os.path.join(self.img_dir, img_name)
        img_vector = cv.imread(img_path)
        if self.transforms:
            transformed = self.transforms(image = img_vector)
            img_vector = transformed['image']
        if self.resize:
            img_vector = cv.resize(img_vector, self.resize)
            img_vector = img_vector.reshape((3, self.resize[0], self.resize[1]))
            
        img_vector = torch.tensor(img_vector)
        return img_vector
        
    def show(self, idx):
        img_vector, target = self.__getitem__(idx)
        plt.imshow(img_vector.detach().numpy())
        plt.title(f'{target.detach().numpy()}')
        plt.show()
        
    def get_class_weights(self):
        class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.array([0, 1, 2, 3, 4]),
                                                          y=self.df['diagnosis'].values)
        class_weights = torch.tensor(class_weights, dtype=torch.float)
        return class_weights



In [5]:
BATCH_SIZE = 16
IMG_DIM = 512

test_dataset = AptosDataset(TEST_PATH,TEST_IMG, 'test', False, resize = (IMG_DIM, IMG_DIM))
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE)
model = resnet50(pretrained=False)
model.fc = nn.Sequential(nn.Linear(in_features=2048, out_features=1024, bias=True),
                         nn.Linear(in_features=1024, out_features=512, bias=True),
                         nn.Linear(in_features=512, out_features=5, bias=True),
                         nn.Softmax())
model.load_state_dict(torch.load('../input/pretrained/model-2.bin')) 
model = model.to(device)
model.eval()
labels = []  
with torch.no_grad():
    for batch, x in enumerate(test_loader):
        output = model(x.float().to(device))

        predictions = output.argmax(dim=1).cpu().detach().tolist() 
        labels.extend(predictions)

FileNotFoundError: [Errno 2] No such file or directory: '../input/pretrained/model-2.bin'

In [6]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
sample_sub['diagnosis'] = labels
sample_sub.to_csv('submission.csv', index = False)

NameError: name 'labels' is not defined